In [2]:
import sys
import os
sys.path.append(os.getcwd())

In [ ]:
import re
from collections import defaultdict, Counter
from docling.document_converter import DocumentConverter

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_CACHE_SYMLINKS"] = "0"

def extract_header_candidates(doc):
    """
    1. Scans 'section_header' items.
    2. Splits merged text (regex).
    3. Returns flat list of candidates with BBOX and SIZE.
    """
    candidates = []

    for item in doc.texts:
        if item.label == "section_header":
            raw_text = item.text.strip()
            if not raw_text: continue

            # Get Metadata
            # We assume the parent BBox applies to the first part
            # For subsequent parts in a split, we theoretically don't have exact X-pos, 
            # but we know they come AFTER.
            prov = item.prov[0] if item.prov else None
            page = prov.page_no if prov else 0
            bbox = prov.bbox if prov else None
            
            # Height = Bottom - Top
            height = abs(bbox.b - bbox.t) if bbox else 0
            
            # Y-Position (Top) - Used to detect "Same Line"
            y_top = round(bbox.t) if bbox else 0
            
            # SPLIT MERGED TEXT
            # "Chapitre I   Section 1" -> ["Chapitre I", "Section 1"]
            split_parts = re.split(r'\n|\s{3,}', raw_text)
            
            for i, part in enumerate(split_parts):
                part = part.strip()
                if not part: continue
                
                # ESTIMATE X-POSITION
                # If it's the 2nd part of a split, it's definitely to the right.
                # We add a fake offset to i > 0 to ensure sorting works.
                x_pos = (bbox.l if bbox else 0) + (i * 100)

                candidates.append({
                    "text": part,
                    "height": round(height, 2),
                    "y_top": y_top,
                    "x_left": x_pos,
                    "page": page
                })

    return candidates

# ==========================================
#  STEP 2: LEARN RULES (Names & Sizes)
# ==========================================
def learn_hierarchy_rules(candidates):
    print("🧠 Learning Document Rules...")
    
    # A. Names (Frequency)
    first_words = []
    for c in candidates:
        tokens = c['text'].lower().split()
        if tokens:
            root = re.sub(r'[^a-zà-ÿ]', '', tokens[0])
            if root: first_words.append(root)
            
    word_counts = Counter(first_words)
    valid_names = {w for w, count in word_counts.items() if count > 2}
    print(f"   ✅ Valid Header Names: {valid_names}")

    # B. Sizes (Frequency)
    raw_sizes = [round(c['height']*2)/2 for c in candidates]
    size_counts = Counter(raw_sizes)
    valid_sizes = sorted([s for s, count in size_counts.items() if count > 2], reverse=True)
    
    # Map Size -> Rank
    size_map = {s: i+1 for i, s in enumerate(valid_sizes)}
    print(f"   ✅ Valid Size Ranks: {size_map}")

    return valid_names, size_map

# ==========================================
#  STEP 3: THE "FIRST IN LINE" FILTER
# ==========================================
def filter_headers(candidates, valid_names, size_map):
    print("🛡️ Applying 'First-in-Line' Filter...")
    
    final_headers = []
    
    # Group by Page -> Y_Position (Lines)
    # Structure: { page_num: { y_coord: [headers...] } }
    lines_map = defaultdict(lambda: defaultdict(list))
    
    for c in candidates:
        lines_map[c['page']][c['y_top']].append(c)

    # PROCESS EACH LINE
    for page in sorted(lines_map.keys()):
        for y_pos in sorted(lines_map[page].keys()):
            items_on_line = lines_map[page][y_pos]
            
            # SORT by X (Left to Right)
            items_on_line.sort(key=lambda x: x['x_left'])
            
            # --- THE RULE ---
            # "Exclude names in the same line, keep only the FIRST"
            first_item = items_on_line[0]
            
            # Validate Name
            tokens = first_item['text'].lower().split()
            if not tokens: continue
            root = re.sub(r'[^a-zà-ÿ]', '', tokens[0])
            
            if root not in valid_names:
                continue # Skip if name is noise (e.g. "Le", "Page")
            
            # Validate Size
            norm_size = round(first_item['height']*2)/2
            if norm_size not in size_map:
                continue # Skip if size is noise
            
            # Assign Rank
            first_item['rank'] = size_map[norm_size]
            
            final_headers.append(first_item)
            
            # DEBUG: Show what we dropped
            if len(items_on_line) > 1:
                dropped = [x['text'] for x in items_on_line[1:]]
                # print(f"   ✂️  Kept: '{first_item['text']}' | Dropped Siblings: {dropped}")

    print(f"✅ Final Clean Headers: {len(final_headers)}")
    return final_headers

# ==========================================
#  MAIN EXECUTION
# ==========================================
def run_pipeline(pdf_path):
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    doc = result.document
    
    # 1. Extract & Split
    candidates = extract_header_candidates(doc)
    
    # 2. Learn
    valid_names, size_map = learn_hierarchy_rules(candidates)
    
    # 3. Filter (The BBox Logic)
    clean_headers = filter_headers(candidates, valid_names, size_map)
    
    # Display Result
    print("\n--- FINAL HIERARCHY ---")
    for h in clean_headers:
        print(f"Rank {h['rank']} | Size {h['height']} | Text: {h['text']}")

# Run
run_pipeline("data/pdfs/random.pdf")

2025-12-29 14:13:16,021 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-29 14:13:16,030 - INFO - Going to convert document batch...
2025-12-29 14:13:16,031 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 5a43216f093a7c32c3d8090bbb471faa
2025-12-29 14:13:16,032 - INFO - Accelerator device: 'cuda:0'
2025-12-29 14:13:17,539 - INFO - Accelerator device: 'cuda:0'
2025-12-29 14:13:18,800 - INFO - Accelerator device: 'cuda:0'
2025-12-29 14:13:19,284 - INFO - Processing document random.pdf
2025-12-29 14:13:21,766 - INFO - Finished converting document random.pdf in 5.75 sec.


🧠 Learning Document Rules...
   ✅ Valid Header Names: {'association'}
   ✅ Valid Size Ranks: {9.0: 1}
🛡️ Applying 'First-in-Line' Filter...
✅ Final Clean Headers: 3

--- FINAL HIERARCHY ---
Rank 1 | Size 8.96 | Text: Association Colistine -T icoplanine
Rank 1 | Size 8.96 | Text: Association Colistine -Rifampicine
Rank 1 | Size 8.96 | Text: Association colistine -acide fusidique
